In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split

In [70]:
data = pd.read_csv("online_retail_II.csv")

In [71]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [72]:
data = data[data["Quantity"] > 0]

In [73]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [74]:
nlargestdata = data.groupby('StockCode')['Quantity'].sum().nlargest(250)
nlargestdata = pd.DataFrame(nlargestdata)
nlargestdata = nlargestdata.reset_index()

In [75]:
nlargestdata.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   StockCode  250 non-null    str  
 1   Quantity   250 non-null    int64
dtypes: int64(1), str(1)
memory usage: 4.0 KB


In [76]:
nlargestdata.shape

(250, 2)

In [77]:
nlargestdata.isnull().sum()

StockCode    0
Quantity     0
dtype: int64

In [78]:
nlargestdata.describe()

,Quantity
count,250.000000
mean,21156.576000
std,16004.118806
min,10128.000000
25%,12687.000000
50%,15804.000000
75%,22655.250000
max,110249.000000


In [79]:
data["Country"].unique()

<StringArray>
[      'United Kingdom',               'France',                  'USA',
              'Belgium',            'Australia',                 'EIRE',
              'Germany',             'Portugal',              'Denmark',
          'Netherlands',               'Poland',      'Channel Islands',
                'Spain',               'Cyprus',               'Greece',
               'Norway',              'Austria',               'Sweden',
 'United Arab Emirates',              'Finland',                'Italy',
          'Switzerland',                'Japan',          'Unspecified',
              'Nigeria',                'Malta',              'Bahrain',
                  'RSA',              'Bermuda',            'Hong Kong',
            'Singapore',             'Thailand',               'Israel',
            'Lithuania',          'West Indies',              'Lebanon',
                'Korea',               'Brazil',               'Canada',
              'Iceland',         'Sau

In [80]:
newdata = data[data["StockCode"].isin(nlargestdata['StockCode'])]

In [81]:
newdata.shape

(282214, 8)

In [82]:
newdata.isnull().sum()

Invoice            0
StockCode          0
Description      149
Quantity           0
InvoiceDate        0
Price              0
Customer ID    52778
Country            0
dtype: int64

In [83]:
newdata = newdata.dropna(subset=["Description"])

In [84]:
newdata.isnull().sum()

Invoice            0
StockCode          0
Description        0
Quantity           0
InvoiceDate        0
Price              0
Customer ID    52629
Country            0
dtype: int64

In [85]:
newdata.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
15,489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01 09:06:00,1.69,13078.0,United Kingdom
25,489436,21181,PLEASE ONE PERSON METAL SIGN,12,2009-12-01 09:06:00,2.10,13078.0,United Kingdom
30,489436,22111,SCOTTIE DOG HOT WATER BOTTLE,24,2009-12-01 09:06:00,4.25,13078.0,United Kingdom


In [86]:
newdata['InvoiceDate'] = pd.to_datetime(newdata['InvoiceDate']).dt.date

In [87]:
newdata.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01,1.25,13085.0,United Kingdom
15,489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01,1.69,13078.0,United Kingdom
25,489436,21181,PLEASE ONE PERSON METAL SIGN,12,2009-12-01,2.10,13078.0,United Kingdom
30,489436,22111,SCOTTIE DOG HOT WATER BOTTLE,24,2009-12-01,4.25,13078.0,United Kingdom


In [88]:
filtereddata = newdata.groupby(["InvoiceDate", "StockCode"])["Quantity"].sum().reset_index()

In [89]:
filtereddata.head()

,InvoiceDate,StockCode,Quantity
0,2009-12-01,15034,3
1,2009-12-01,15036,55
2,2009-12-01,15056N,30
3,2009-12-01,16047,1
4,2009-12-01,16156S,25


In [90]:
filtereddata.shape

(92673, 3)

In [91]:
def fill_dates(group):
    date_range = pd.date_range(group['InvoiceDate'].min(), group['InvoiceDate'].max(), freq='D')

    group = group.set_index('InvoiceDate').reindex(date_range, fill_value=0)
    group.index.name = 'InvoiceDate'
    return group

full_data = filtereddata.groupby('StockCode').apply(fill_dates).reset_index()


In [92]:
full_data

,StockCode,InvoiceDate,Quantity
0,15034,2009-12-01,3
1,15034,2009-12-02,0
2,15034,2009-12-03,0
3,15034,2009-12-04,0
4,15034,2009-12-05,0
...,...,...,...
159904,POST,2011-12-05,15
159905,POST,2011-12-06,25
159906,POST,2011-12-07,21
159907,POST,2011-12-08,12


In [93]:
full_data = full_data.sort_values(['StockCode', 'InvoiceDate'])

In [94]:
full_data

,StockCode,InvoiceDate,Quantity
0,15034,2009-12-01,3
1,15034,2009-12-02,0
2,15034,2009-12-03,0
3,15034,2009-12-04,0
4,15034,2009-12-05,0
...,...,...,...
159904,POST,2011-12-05,15
159905,POST,2011-12-06,25
159906,POST,2011-12-07,21
159907,POST,2011-12-08,12


In [95]:
full_data['lag_1'] = full_data.groupby('StockCode')['Quantity'].shift(1)

In [96]:
full_data.head()

,StockCode,InvoiceDate,Quantity,lag_1
0,15034,2009-12-01,3,NaN
1,15034,2009-12-02,0,3.0
2,15034,2009-12-03,0,0.0
3,15034,2009-12-04,0,0.0
4,15034,2009-12-05,0,0.0


In [97]:
full_data['lag_7'] = full_data.groupby('StockCode')['Quantity'].shift(7)
full_data['rolling_mean'] = full_data.groupby('StockCode')['Quantity'].transform(lambda x: x.shift(1).rolling(7).mean())

In [98]:
full_data.head(10)

,StockCode,InvoiceDate,Quantity,lag_1,lag_7,rolling_mean
0,15034,2009-12-01,3,NaN,NaN,NaN
1,15034,2009-12-02,0,3.0,NaN,NaN
2,15034,2009-12-03,0,0.0,NaN,NaN
3,15034,2009-12-04,0,0.0,NaN,NaN
4,15034,2009-12-05,0,0.0,NaN,NaN
5,15034,2009-12-06,0,0.0,NaN,NaN
6,15034,2009-12-07,3,0.0,NaN,NaN
7,15034,2009-12-08,0,3.0,3.0,0.857143
8,15034,2009-12-09,3,0.0,0.0,0.428571
9,15034,2009-12-10,0,3.0,0.0,0.857143


In [99]:
full_data = full_data.dropna()

In [100]:
full_data


,StockCode,InvoiceDate,Quantity,lag_1,lag_7,rolling_mean
7,15034,2009-12-08,0,3.0,3.0,0.857143
8,15034,2009-12-09,3,0.0,0.0,0.428571
9,15034,2009-12-10,0,3.0,0.0,0.857143
10,15034,2009-12-11,0,0.0,0.0,0.857143
11,15034,2009-12-12,0,0.0,0.0,0.857143
...,...,...,...,...,...,...
159904,POST,2011-12-05,15,13.0,20.0,15.571429
159905,POST,2011-12-06,25,15.0,22.0,14.857143
159906,POST,2011-12-07,21,25.0,25.0,15.285714
159907,POST,2011-12-08,12,21.0,12.0,14.714286


In [101]:
full_data.shape

(158165, 6)

In [102]:
full_data['day_of_week'] = pd.to_datetime(full_data["InvoiceDate"]).dt.day_of_week
full_data['month'] = pd.to_datetime(full_data['InvoiceDate']).dt.month

In [103]:
full_data.head()

,StockCode,InvoiceDate,Quantity,lag_1,lag_7,rolling_mean,day_of_week,month
7,15034,2009-12-08,0,3.0,3.0,0.857143,1,12
8,15034,2009-12-09,3,0.0,0.0,0.428571,2,12
9,15034,2009-12-10,0,3.0,0.0,0.857143,3,12
10,15034,2009-12-11,0,0.0,0.0,0.857143,4,12
11,15034,2009-12-12,0,0.0,0.0,0.857143,5,12


In [104]:
full_data.tail()

,StockCode,InvoiceDate,Quantity,lag_1,lag_7,rolling_mean,day_of_week,month
159904,POST,2011-12-05,15,13.0,20.0,15.571429,0,12
159905,POST,2011-12-06,25,15.0,22.0,14.857143,1,12
159906,POST,2011-12-07,21,25.0,25.0,15.285714,2,12
159907,POST,2011-12-08,12,21.0,12.0,14.714286,3,12
159908,POST,2011-12-09,10,12.0,17.0,14.714286,4,12


In [105]:
full_data.info()

<class 'pandas.DataFrame'>
Index: 158165 entries, 7 to 159908
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype        
---  ------        --------------   -----        
 0   StockCode     158165 non-null  str          
 1   InvoiceDate   158165 non-null  datetime64[s]
 2   Quantity      158165 non-null  int64        
 3   lag_1         158165 non-null  float64      
 4   lag_7         158165 non-null  float64      
 5   rolling_mean  158165 non-null  float64      
 6   day_of_week   158165 non-null  int32        
 7   month         158165 non-null  int32        
dtypes: datetime64[s](1), float64(3), int32(2), int64(1), str(1)
memory usage: 9.7 MB


In [106]:
full_data.iloc[110715]

StockCode                     22659
InvoiceDate     2011-08-31 00:00:00
Quantity                         14
lag_1                           6.0
lag_7                          11.0
rolling_mean               3.714286
day_of_week                       2
month                             8
Name: 111870, dtype: object

In [110]:
Train = full_data[full_data['InvoiceDate'] < '2011-08-31']
Test = full_data[full_data['InvoiceDate'] >= '2011-08-31']

In [112]:
Train.shape

(134992, 8)

In [113]:
Test.shape

(23173, 8)

In [124]:
X_train = Train.drop(['Quantity','StockCode', 'InvoiceDate'], axis=1)

In [125]:
X_train.head

<bound method NDFrame.head of         lag_1  lag_7  rolling_mean  day_of_week  month
7         3.0    3.0      0.857143            1     12
8         0.0    0.0      0.428571            2     12
9         3.0    0.0      0.857143            3     12
10        0.0    0.0      0.857143            4     12
11        0.0    0.0      0.857143            5     12
...       ...    ...           ...          ...    ...
159803   22.0   22.0     12.142857            4      8
159804   12.0    0.0     10.714286            5      8
159805    0.0    5.0     10.714286            6      8
159806    0.0   10.0     10.000000            0      8
159807    0.0   12.0      8.571429            1      8

[134992 rows x 5 columns]>

In [126]:
Y_train = Train['Quantity']

In [127]:
Y_train.head()

7     0
8     3
9     0
10    0
11    0
Name: Quantity, dtype: int64